# 01 — Preprocessing

Prepares the Ministry of Commerce (MC) active commercial registrations for the survival model.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

## Part 1 — Did the ID change, or only its column name?

Up to 2025 Q3 the ID column is called **الرقم الوطني**. From 2025 Q4 it is called **الرقم الموحد**.
This part checks whether the numbers stayed the same, by comparing **2025 Q2** (the last complete quarter
with the old name) with **2025 Q4** (the first quarter with the new name).
2025 Q3 is skipped on purpose: most of its rows are missing and half of it has no ID.

### Cell 1 — Setup
Finds the data. On Colab it downloads the files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))

MC_DIR = DATA_DIR / 'MC'
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — The files this part needs
Only the 4 files of 2025 Q2 and 2025 Q4 (each quarter comes in two parts).
The IDs are the files' Google Drive IDs, used only when running on Colab.

In [ ]:
FILES = {
    '2025Q2': [('2025-02q-active-crs_PART1-2.xlsx', '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
               ('2025-02q-active-crs_PART2.xlsx',   '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_')],
    '2025Q4': [('2025-04q-active-crs_PART1.xlsx',   '1tzttjeDaz8WQrFSGE7WYI5AMQiNSsRrM'),
               ('2025-04q-active-crs_PART2.xlsx',   '1rqw-5VqvreNlA7hJN4sY3wOzgmOI485a')],
}

if IN_COLAB:
    import gdown
    MC_DIR.mkdir(parents=True, exist_ok=True)
    for parts in FILES.values():
        for name, drive_id in parts:
            if not (MC_DIR / name).exists():
                gdown.download(id=drive_id, output=str(MC_DIR / name), quiet=True)

for quarter, parts in FILES.items():
    for name, _ in parts:
        path = MC_DIR / name
        print(quarter, name, 'found' if path.exists() else 'MISSING', f'({path.stat().st_size / 1e6:.0f} MB)' if path.exists() else '')

### Cell 3 — Read only the columns the test needs
ID, name, start date and city. The two quarters use different column names, so each is renamed
to the same four names: `id`, `name`, `start_date`, `city`. Everything is read as text, so a number
like `7001234567` is never changed (no rounding, no lost zeros).

In [ ]:
import pandas as pd

COLUMNS = {
    # old names (up to 2025 Q3)
    '2025Q2': {'الرقم الوطني': 'id', 'اسم السجل التجاري': 'name',
               'تاريخ الإصدار (ميلادي)': 'start_date', 'اسم المدينة (بالعربية)': 'city'},
    # new names (from 2025 Q4)
    '2025Q4': {'الرقم الموحد': 'id', 'اسم السجل': 'name',
               'تاريخ انشاء السجل': 'start_date', 'المدينة': 'city'},
}

def read_quarter(quarter):
    tables = []
    for name, _ in FILES[quarter]:
        part = pd.read_excel(MC_DIR / name, engine='calamine', dtype=str,
                             usecols=list(COLUMNS[quarter]))
        tables.append(part.rename(columns=COLUMNS[quarter]).assign(source_file=name))
    table = pd.concat(tables, ignore_index=True)
    for column in ['id', 'name', 'city', 'start_date']:
        table[column] = table[column].str.strip()
    table['start_day'] = table['start_date'].str[:10]   # date only, without the time
    return table

q2 = read_quarter('2025Q2')
q4 = read_quarter('2025Q4')
print(f'2025 Q2: {len(q2):,} rows | 2025 Q4: {len(q4):,} rows')

### Cell 4 — Compare the IDs (numbers)
1. How many IDs each quarter has, and whether any ID repeats inside a quarter.
2. How many 2025 Q2 IDs also appear in 2025 Q4.
3. For the IDs found in both: do the name, start date and city also agree? If they do, it is the same business
   and only the column name changed.

In [ ]:
def describe(label, table):
    ids = table['id']
    print(f'{label}: {len(table):,} rows | empty ID: {ids.isna().sum():,} | '
          f'unique IDs: {ids.nunique():,} | IDs that repeat: {ids.duplicated(keep=False).sum():,} rows')
    print(f'   IDs starting with 7: {ids.str.startswith("7").mean():.1%} | 10 digits long: {ids.str.fullmatch(r"\d{10}").mean():.1%}')

describe('2025 Q2', q2)
describe('2025 Q4', q4)

q2_ids, q4_ids = set(q2['id'].dropna()), set(q4['id'].dropna())
both = q2_ids & q4_ids
print()
print(f'IDs in both quarters: {len(both):,}')
print(f'   = {len(both) / len(q2_ids):.1%} of the 2025 Q2 IDs')
print(f'   = {len(both) / len(q4_ids):.1%} of the 2025 Q4 IDs')

# For the shared IDs, compare the details (first row of each ID, to keep it simple)
pairs = (q2.drop_duplicates('id').set_index('id')[['name', 'start_day', 'city']]
         .join(q4.drop_duplicates('id').set_index('id')[['name', 'start_day', 'city']],
               how='inner', lsuffix='_q2', rsuffix='_q4'))
print()
print('For the IDs in both quarters, how often the details agree:')
for column in ['name', 'start_day', 'city']:
    print(f'   same {column:<10}: {(pairs[column + "_q2"] == pairs[column + "_q4"]).mean():.1%}')


### Cell 5 — Twenty random examples
Twenty IDs found in both quarters, side by side (2025 Q2 on the left, 2025 Q4 on the right),
to look at with our own eyes.

In [ ]:
pairs.sample(20, random_state=1)

## Part 1b — Same business, different ID?

### Cell 6 — Missing businesses that may have come back under a new ID
About 109,000 businesses of 2025 Q2 are not in 2025 Q4. Most probably closed, but some may have
re-registered with a **new** ID (for example a مؤسسة that became a شركة). If we do not catch them, they
would wrongly count as businesses that did not survive.

This cell only **counts and shows examples**. It links nothing. Two ways to look for a possible new ID:
- **A (strict):** a new 2025 Q4 ID with the same name, start date and city.
- **B (looser):** a new 2025 Q4 ID with the same name once the legal form (شركة، مؤسسة، المحدودة…) is removed,
  in the same city. A re-registered business often gets a new start date, so B does not require the same date.

"More than one candidate" means a match is unsure: those must never be linked automatically.

In [ ]:
import re

# Businesses in only one of the two quarters
missing = q2[~q2['id'].isin(q4_ids)].drop_duplicates('id')    # in 2025 Q2, not in 2025 Q4
new_q4  = q4[~q4['id'].isin(q2_ids)]                           # in 2025 Q4, not in 2025 Q2
print(f'In 2025 Q2 but not in 2025 Q4: {len(missing):,}')
print(f'In 2025 Q4 but not in 2025 Q2: {len(new_q4):,}')

def core_name(name):
    """The name without its legal form and spelling differences, for comparing only:
    'شركة الوليد صالح الحربي المحدودة' and 'مؤسسة الوليد صالح الحربي' both become 'الوليد صالح الحربي'."""
    t = str(name)
    t = re.sub('[إأآ]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    t = re.sub(r'\b(شركه|مؤسسه|المحدوده|محدوده|ذات مسؤوليه|شخص واحد|مساهمه|مبسطه|فرع)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

missing = missing.assign(core=missing['name'].map(core_name))
new_q4  = new_q4.assign(core=new_q4['name'].map(core_name))

def candidates(keys, label):
    found = missing.merge(new_q4, on=keys, suffixes=('_q2', '_q4'))
    per_business = found.groupby('id_q2')['id_q4'].nunique()
    print(f'{label}:')
    print(f'   missing Q2 businesses with a possible new ID: {len(per_business):,} '
          f'(exactly one candidate: {(per_business == 1).sum():,} | more than one: {(per_business > 1).sum():,})')
    return found

print()
strict = candidates(['name', 'start_day', 'city'], 'A. Same name + same start date + same city')
loose  = candidates(['core', 'city'], 'B. Same name without the legal form + same city (any start date)')

# Examples to look at with our own eyes
show = ['id_q2', 'name_q2', 'start_day_q2', 'id_q4', 'name_q4', 'start_day_q4', 'city']
print()
print('Examples of B (10 random):')
loose[show].sample(min(10, len(loose)), random_state=1)


### Cell 7 — The ID that repeats in 2025 Q2
The two rows that share one ID, to see if they are a duplicate or two different records.
Nothing is deleted: a duplicate will be flagged, not dropped.

In [ ]:
repeated = q2[q2['id'].duplicated(keep=False)]
print(f'Rows whose ID repeats in 2025 Q2: {len(repeated)}')
repeated[['id', 'name', 'start_date', 'city', 'source_file']]


## Part 2 — All nine quarters in one shape

### Cell 8 — Read every quarter
Reads all 18 files (9 quarters, 2 parts each) with **all** their columns. Column names change over the years,
so each one gets a single common name (for example "تاريخ الإصدار (ميلادي)" and "تاريخ انشاء السجل" both become
`start_date`). Two columns say where a row came from: `quarter` and `source_file`.

2025 Q3 is read too. Its PART1 has no ID, so those rows get the flag `flag_no_id`. Nothing is deleted.
Takes about 10–15 minutes (download and about 13 million rows).

In [ ]:
import pandas as pd

# Every quarter: (quarter, file name, Google Drive ID)
ALL_FILES = [
    ('2024Q3', 'CR_2024-03q-active-crs_PART1-2.xlsx', '1YAjKdTEzTf66Z_wK0nNVAboD_NoskbKV'),
    ('2024Q3', 'CR_2024-03q-active-crs_PART2-2.xlsx', '1bCd3HFyhCk9VbZgT79wxsIxgBBflKps0'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART1.xlsx',   '1GPJUCZ5HiTa-_ZJBVLjrp58Ha53nOBGN'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART2.xlsx',   '10y-NeQI6P78Sb_b3mqjTjcdpZq9zNvNK'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART1.xlsx',   '1QTeifZpbmotbWmbZoliWoVcDxSK8y8uf'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART2.xlsx',   '12LqZsJ8WHI4qUza_mXP4iUQPuwG9x_2d'),
    ('2025Q2', '2025-02q-active-crs_PART1-2.xlsx',    '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
    ('2025Q2', '2025-02q-active-crs_PART2.xlsx',      '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_'),
    ('2025Q3', '2025-03q-active-crs_PART1.xlsx',      '11TR0gdOc1u486yR8y9jYkA5jXqwLwxVo'),
    ('2025Q3', '2025-03q-active-crs_PART2-2.xlsx',    '1K4jyfIVbWfALc8bdB0-agPAwBev6SqFD'),
    ('2025Q4', '2025-04q-active-crs_PART1.xlsx',      '1tzttjeDaz8WQrFSGE7WYI5AMQiNSsRrM'),
    ('2025Q4', '2025-04q-active-crs_PART2.xlsx',      '1rqw-5VqvreNlA7hJN4sY3wOzgmOI485a'),
    ('2026Q1', '2026 01q active_PART1.xlsx',          '1uXVoAo8IoDVhZEBJcZEdq1T3Rhfkw9kZ'),
    ('2026Q1', '2026 01q active_PART2.xlsx',          '1YukIJGFb8GIBxV5rCxXmXN023kU6bjr2'),
    ('2026Q2', '2026 02q active crs_PART1.xlsx',      '1KBjgL6FXdI-aeWumy15rnhlE20_W0ZNJ'),
    ('2026Q2', '2026 02q active crs_PART2.xlsx',      '1BrmeQHLyxET9xPulJDxBQ_68qZ8zu6kf'),
    ('2026Q3', 'CR A 2026-09-30_PART1.xlsx',          '1TGnGo3x2rkvDvbwREznik4ko0WGzXhHy'),
    ('2026Q3', 'CR A 2026-09-30_PART2-2.xlsx',        '1siJD5yNZTXRINRazJP2LfpFm8xlKcVZw'),
]

# One common name for every column, whatever its name in a given year
COMMON_NAMES = {
    # ID of the record
    'الرقم الوطني': 'id',                          # up to 2025 Q3
    'الرقم الموحد': 'id',                          # from 2025 Q4
    # for a branch: the ID of its main record (only up to 2025 Q3)
    'الرقم الرئيسي للسجل التجاري': 'main_id',
    # the commercial registration number (only from 2025 Q4)
    'رقم السجل': 'cr_number',
    # main record or branch
    'نوع السجل التجاري': 'record_type',
    'نوع السجل': 'record_type',
    'اسم السجل التجاري': 'name',
    'اسم السجل': 'name',
    # establishment, company ...
    'نوع الكيان': 'entity_type',
    'الكيان القانوني': 'entity_type',
    # the detailed legal form (despite its Arabic name, this is not the business activity)
    'نوع النشاط التجاري (بالعربية)': 'legal_form',
    'الكيان القانوني.1': 'legal_form',
    'رأس المال': 'capital',
    'اسم المدينة (بالعربية)': 'city',
    'المدينة': 'city',
    'اسم المنطقة (بالعربية)': 'region',
    'المنطقة': 'region',
    'مفصح بالتجارة الإلكترونية؟': 'ecommerce',
    'مفصح بتجارة إلكترونية؟': 'ecommerce',
    'مفصح تجارة إلكترونية؟': 'ecommerce',
    'تاريخ الإصدار (ميلادي)': 'start_date',
    'تاريخ انشاء السجل': 'start_date',
}
ORDER = ['quarter', 'source_file', 'id', 'main_id', 'cr_number', 'record_type', 'name', 'entity_type',
         'legal_form', 'capital', 'region', 'city', 'ecommerce', 'start_date']
# Columns with few different values are stored as categories, and text in a compact format, to save memory
FEW_VALUES = ['quarter', 'source_file', 'record_type', 'entity_type', 'legal_form', 'region', 'city', 'ecommerce']

if IN_COLAB:
    import gdown
    MC_DIR.mkdir(parents=True, exist_ok=True)
    for quarter, name, drive_id in ALL_FILES:
        if not (MC_DIR / name).exists():
            print('downloading', name)
            gdown.download(id=drive_id, output=str(MC_DIR / name), quiet=True)

unknown_columns = set()
parts = []
for quarter, name, _ in ALL_FILES:
    part = pd.read_excel(MC_DIR / name, engine='calamine', dtype=str)
    unknown_columns |= set(part.columns) - set(COMMON_NAMES)
    part = part.rename(columns=COMMON_NAMES)
    for column in part.columns:
        part[column] = part[column].str.strip()
    part = part.assign(quarter=quarter, source_file=name).reindex(columns=ORDER)
    for column in ORDER:
        part[column] = part[column].astype('category' if column in FEW_VALUES else 'string[pyarrow]')
    parts.append(part)
    print(f'{quarter}  {name:<40} {len(part):>10,} rows')

mc = pd.concat(parts, ignore_index=True)
for column in FEW_VALUES:
    mc[column] = mc[column].astype('category')
del parts

# Rows with no ID (2025 Q3 PART1) get a flag. Nothing is deleted.
mc['flag_no_id'] = mc['id'].isna()

print()
print(f'All quarters together: {len(mc):,} rows | memory used: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print('Columns the notebook did not recognise:', sorted(unknown_columns) or 'none')


### Cell 9 — Health check per quarter
Only counts. For every quarter: rows, main records and branches, rows with no ID, IDs that repeat
(and how many of those appear in both PART1 and PART2, like the overlap found in 2025 Q2), and which
columns the quarter does not have at all.

In [ ]:
def quarter_health(table):
    rows = []
    for quarter, q in table.groupby('quarter', observed=True):
        with_id = q[q['id'].notna()]
        per_id = with_id.groupby('id', observed=True)['source_file'].agg(['size', 'nunique'])
        rows.append({
            'quarter': quarter,
            'rows': len(q),
            'main': (q['record_type'] == 'رئيسي').sum(),
            'branch': (q['record_type'] == 'فرعي').sum(),
            'type unknown': q['record_type'].isna().sum(),
            'no ID': q['id'].isna().sum(),
            'IDs repeated': int((per_id['size'] > 1).sum()),
            '…in both parts': int(((per_id['size'] > 1) & (per_id['nunique'] > 1)).sum()),
            'columns missing': ', '.join(c for c in ['main_id', 'cr_number', 'legal_form', 'ecommerce', 'region']
                                         if q[c].isna().all()) or '—',
        })
    return pd.DataFrame(rows).set_index('quarter')

health = quarter_health(mc)
print(health.drop(columns='columns missing').to_string())
print()
print(health[['columns missing']].to_string())


### Cell 10 — Can we trust each quarter?
Two checks that only count.

**A. Main / branch labels.** Up to 2025 Q3 a branch has a `main_id` that points to its main record. Counts how
often `record_type` agrees with that, to see whether a quarter (such as 2025 Q1, almost all "main") labels
branches as main.

**B. Completeness.** A business that is active in a later quarter and already existed before the earlier quarter
ended should also appear in the earlier quarter. The share that does = how complete the earlier quarter is.
A low share means businesses are missing from that quarter, and would wrongly look closed in the labels.

In [ ]:
QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
QUARTER_END = {'2024Q3': '2024-09-30', '2024Q4': '2024-12-31', '2025Q1': '2025-03-31', '2025Q2': '2025-06-30',
               '2025Q3': '2025-09-30', '2025Q4': '2025-12-31', '2026Q1': '2026-03-31', '2026Q2': '2026-06-30',
               '2026Q3': '2026-09-30'}
start_day = mc['start_date'].str[:10]          # date only; dates are written as YYYY-MM-DD, so text order = date order

# ---------- A. Is "main / branch" labelled correctly? (quarters that have main_id) ----------
print('A. Main / branch labels compared with main_id (a real branch points to another record)')
rows = []
for quarter in QUARTERS[:5]:
    q = mc[mc['quarter'] == quarter]
    points_elsewhere = (q['main_id'].notna() & (q['main_id'] != q['id'])).fillna(False)
    is_main, is_branch = q['record_type'] == 'رئيسي', q['record_type'] == 'فرعي'
    rows.append({'quarter': quarter,
                 'main': int(is_main.sum()),
                 'main that points to another record': int((is_main & points_elsewhere).sum()),
                 'branch': int(is_branch.sum()),
                 'branch with no main record': int((is_branch & ~points_elsewhere).sum())})
print(pd.DataFrame(rows).set_index('quarter').to_string())

# ---------- B. How complete is each quarter? ----------
# Main records of a later quarter that already existed before the earlier quarter ended
# should (almost) all be in the earlier quarter too. The share found = how complete the earlier quarter is.
print()
print('B. Completeness of each quarter (checked against the quarter after it)')
pairs = list(zip(QUARTERS, QUARTERS[1:])) + [('2025Q2', '2025Q4')]   # 2025Q2 -> 2025Q4 skips the broken 2025Q3
rows = []
for earlier, later in pairs:
    later_main = mc[(mc['quarter'] == later) & (mc['record_type'] == 'رئيسي')]
    existed = later_main[(start_day[later_main.index] <= QUARTER_END[earlier]).fillna(False)]   # empty date = not counted
    earlier_ids = mc.loc[(mc['quarter'] == earlier) & mc['id'].notna(), 'id']
    found = existed['id'].isin(earlier_ids)
    rows.append({'earlier quarter': earlier, 'checked against': later,
                 'main records that already existed': len(existed),
                 'found in earlier quarter': int(found.sum()),
                 'completeness': f'{found.mean():.1%}'})
print(pd.DataFrame(rows).set_index('earlier quarter').to_string())


## Part 3 — Cohorts, 2025 Q3, and ID matching

### Cell 11 — How big is each cohort? (only counts)
A cohort is all main records that started in the same quarter. For each cohort: how many businesses we see,
whether the quarter it started in is trustworthy, and whether the quarter one year later (needed for the
1-year label) is trustworthy.

In [ ]:
# Quarters whose file is complete enough to say "this business is gone" (from Cell 10)
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']

mains = mc[(mc['record_type'] == 'رئيسي') & mc['id'].notna()]
cohort = pd.to_datetime(mains['start_date'].str[:10], errors='coerce').dt.to_period('Q').astype(str)
mains = mains.assign(cohort=cohort)            # cohort = the quarter the business started in
businesses = mains.drop_duplicates('id')       # each business once

rows = []
for quarter in QUARTERS:
    one_year_later = str(pd.Period(quarter, 'Q') + 4)
    rows.append({
        'cohort (start quarter)': quarter,
        'businesses (seen in any quarter)': int((businesses['cohort'] == quarter).sum()),
        'seen in their own quarter': int(mains.loc[(mains['quarter'] == quarter) & (mains['cohort'] == quarter), 'id'].nunique()),
        'own quarter trustworthy': 'yes' if quarter in TRUSTED else 'no',
        '1-year check in': one_year_later,
        '1-year quarter trustworthy': 'yes' if one_year_later in TRUSTED else 'no (or no data yet)',
    })
print(pd.DataFrame(rows).set_index('cohort (start quarter)').to_string())
print()
print(f'Businesses that started before 2024 Q3: {int((businesses["cohort"] < "2024Q3").sum()):,}')
print(f'Businesses with no readable start date: {int((businesses["cohort"] == "NaT").sum()):,}')


### Cell 12 — What is 2025 Q3 PART1? (only counts)
The 109,541 rows with no ID: which start dates they cover, their capital and entity type, and whether each
row can be found in 2025 Q2 or 2025 Q4 by the same name, start date and city (if so, its ID can be recovered).

In [ ]:
p1 = mc[(mc['quarter'] == '2025Q3') & mc['id'].isna()]
p1_day = p1['start_date'].str[:10]

print(f'2025 Q3 PART1: {len(p1):,} rows (no ID)')
print(f'Start dates from {p1_day.min()} to {p1_day.max()}')
print()
print('Rows per start month:')
print(p1_day.str[:7].value_counts().sort_index().to_string())
print()
print('Capital (most common values):')
print(p1['capital'].value_counts().head(5).to_string())
print()
print('Entity type:')
print(p1['entity_type'].value_counts().to_string())

# Can each row be found in another quarter by name + start date + city? (If yes, its ID can be recovered.)
left = p1.assign(start_day=p1_day)[['name', 'start_day', 'city']].reset_index()

def find_in(quarter):
    other = mc.loc[(mc['quarter'] == quarter) & mc['id'].notna(), ['id', 'name', 'start_date', 'city']]
    other = other.assign(start_day=other['start_date'].str[:10])[['id', 'name', 'start_day', 'city']]
    found = left.merge(other, on=['name', 'start_day', 'city'], how='left')
    ids_per_row = found.groupby('index')['id'].nunique()
    print(f'Found in {quarter}: exactly one ID: {(ids_per_row == 1).sum():,} | '
          f'more than one: {(ids_per_row > 1).sum():,} | not found: {(ids_per_row == 0).sum():,}')

print()
find_in('2025Q2')
find_in('2025Q4')


### Cell 13 — ID matching (adds two columns, deletes nothing)
The agreed rule:
1. Same ID = same business.
2. An old ID that disappears and never comes back is linked to a new ID only if: same name once the legal
   form is removed, same city, the new start date is later, and there is exactly one candidate **both ways**
   (the new ID is not claimed by another old ID either). Only between trustworthy quarters.
3. Unsure cases are never linked: flagged `ambiguous_not_linked`.

Adds `entity_id` (one ID per business across all quarters) and `link_flag`
(`same_id`, `linked_new_id`, `ambiguous_not_linked`, `duplicate_across_parts`, `no_id`).

After a restart, run only Cell 1, Cell 8 and this cell (the Part 1 cells are not needed and use memory).

In [ ]:
import gc
import re
import numpy as np

# Free memory: the test tables of Part 1 and the checks we no longer need
for name in ['q2', 'q4', 'pairs', 'missing', 'new_q4', 'strict', 'loose', 'p1', 'p1_day', 'left',
             'mains', 'businesses', 'cohort', 'present', 'start_day', 'repeated', 'health', 'links']:
    globals().pop(name, None)
gc.collect()

QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']     # complete quarters (Cell 10)

def core_name(name):
    """The name without its legal form and spelling differences, for comparing only:
    'شركة الوليد صالح الحربي المحدودة' and 'مؤسسة الوليد صالح الحربي' both become 'الوليد صالح الحربي'."""
    t = str(name)
    t = re.sub('[إأآ]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    t = re.sub(r'\b(شركه|مؤسسه|المحدوده|محدوده|ذات مسؤوليه|شخص واحد|مساهمه|مبسطه|فرع)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

is_main = mc['record_type'] == 'رئيسي'
has_id = mc['id'].notna()

# The IDs of each quarter, each ID once
ids_in = {q: pd.Index(mc.loc[(mc['quarter'] == q) & has_id, 'id'].unique()) for q in QUARTERS}

def ids_in_any(quarters):
    out = ids_in[quarters[0]]
    for q in quarters[1:]:
        out = out.union(ids_in[q])
    return out

def main_rows(quarter, ids):
    rows = mc.loc[(mc['quarter'] == quarter) & is_main, ['id', 'name', 'city', 'start_date']]
    rows = rows[rows['id'].isin(ids)].drop_duplicates('id')
    return rows.assign(day=rows['start_date'].str[:10], core=rows['name'].map(core_name)).drop(columns='start_date')

# Links old ID -> new ID, only between trustworthy quarters that follow each other
found_links, ambiguous, summary = [], set(), []
for i, (old_q, new_q) in enumerate(zip(TRUSTED, TRUSTED[1:])):
    vanished = ids_in[old_q].difference(ids_in_any(TRUSTED[i + 1:]))                     # gone, and never back
    appeared = ids_in[new_q].difference(ids_in_any(QUARTERS[:QUARTERS.index(old_q) + 1]))  # new, never seen before
    old, new = main_rows(old_q, vanished), main_rows(new_q, appeared)

    pairs = old.merge(new, on=['core', 'city'], suffixes=('_old', '_new'))
    pairs = pairs[(pairs['day_new'] > pairs['day_old']).fillna(False)]   # the new registration is later (empty date = no)
    per_old = pairs.groupby('id_old')['id_new'].transform('nunique')
    per_new = pairs.groupby('id_new')['id_old'].transform('nunique')
    good = pairs[(per_old == 1) & (per_new == 1)]                         # one candidate, both ways
    unsure = set(pairs['id_old']) - set(good['id_old'])
    ambiguous |= unsure
    found_links.append(good.assign(old_quarter=old_q, new_quarter=new_q))
    summary.append({'from': old_q, 'to': new_q, 'gone (main)': len(old), 'new (main)': len(new),
                    'linked': len(good), 'unsure, not linked': len(unsure)})
    del vanished, appeared, old, new, pairs

links = pd.concat(found_links, ignore_index=True)
del ids_in, found_links
gc.collect()

# One entity_id per business: a linked new ID takes the entity_id of the old one (also through chains).
# Only the linked rows change; every other row keeps its own ID.
entity_of = {}
for old_id, new_id in zip(links['id_old'], links['id_new']):
    entity_of[new_id] = entity_of.get(old_id, old_id)
mc['entity_id'] = mc['id']
linked_rows = mc['id'].isin(list(entity_of)).to_numpy()
mc.loc[linked_rows, 'entity_id'] = mc.loc[linked_rows, 'id'].map(entity_of)

# How each row was linked. Nothing is deleted.
FLAGS = ['same_id', 'linked_new_id', 'ambiguous_not_linked', 'duplicate_across_parts', 'no_id']
codes = np.zeros(len(mc), dtype=np.int8)                       # 0 = same_id
codes[linked_rows] = 1
codes[mc['id'].isin(list(ambiguous)).to_numpy()] = 2
for quarter in QUARTERS:                                       # the 2nd copy of an ID inside one quarter
    in_quarter = np.flatnonzero((mc['quarter'] == quarter).to_numpy())
    ids = mc['id'].iloc[in_quarter]
    codes[in_quarter[(ids.notna() & ids.duplicated()).to_numpy()]] = 3
codes[(~has_id).to_numpy()] = 4
mc['link_flag'] = pd.Categorical.from_codes(codes, FLAGS)
del codes
gc.collect()

print(pd.DataFrame(summary).set_index('from').to_string())
print()
print('Rows per link_flag:')
print(mc['link_flag'].value_counts().to_string())
print()
print(f'Different IDs: {mc["id"].nunique():,} | different businesses (entity_id): {mc["entity_id"].nunique():,}')
print(f'Memory used by the data: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print()
print('10 random links:')
links[['old_quarter', 'id_old', 'name_old', 'day_old', 'id_new', 'name_new', 'day_new', 'city']].sample(min(10, len(links)), random_state=1)


## Part 4 — The one-year survival label

### Cell 14 — Build the label (one row per business)
**Method**
1. **Unit:** one business = one `entity_id` (from Cell 13), main records only. A business that re-registered
   under a new ID counts once. The duplicated row and rows with no ID are not counted.
2. **Cohort:** the quarter of the business's start date (the original business's date if it re-registered).
3. **One-year check:** the quarter 4 quarters later (cohort 2025 Q2 → 2026 Q2).
4. **Label:** `survived_1y` = 1 if the business is in the one-year quarter (under any of its IDs),
   0 if it is not (it disappeared; the files list only active registrations, so the reason is unknown).
5. **One row per business** in the table `labels`, with `cohort`, `cohort_quality`, `survived_1y` and
   `label_reason` (`present_after_1_year`, `came_back_with_new_id`, `disappeared`,
   `missing_at_1_year_back_later`, `cannot_judge`).

Only cohorts whose one-year quarter is trustworthy get a label; `cohort_quality` says how much the start of the
cohort can be trusted (`clean`, `start_incomplete`, `start_missing`). Every other business stays in the table
with `cannot_judge`. A business missing at the check but back in a later quarter is left undecided for now.

Needs Cell 1, Cell 8 and Cell 13 run first.

In [ ]:
import gc
import numpy as np

QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']     # complete quarters (Cell 10)

# Cohorts that get a label, and how much we trust them (the 1-year quarter of each is trustworthy)
COHORT_QUALITY = {
    '2025Q2': 'clean',              # start file and 1-year file both complete
    '2025Q1': 'start_incomplete',   # start file ~81% complete
    '2024Q4': 'start_incomplete',   # start file ~63% complete
    '2025Q3': 'start_missing',      # start file has none of the new businesses
}

# 1. The unit: one business = one entity_id, main records only.
#    Not counted: the 2nd copy of a duplicated row, and rows with no ID.
use = mc.loc[(mc['record_type'] == 'رئيسي')
             & mc['link_flag'].isin(['same_id', 'linked_new_id', 'ambiguous_not_linked']),
             ['entity_id', 'id', 'quarter', 'start_date', 'link_flag']]
use = use.assign(start_day=use['start_date'].str[:10]).drop(columns='start_date')

# One row per business. Its start date is the earliest start date of its IDs
# (for a re-registered business, that is the original business's date).
labels = use.groupby('entity_id', observed=True).agg(start_day=('start_day', 'min'))
labels['ambiguous_link'] = labels.index.isin(use.loc[use['link_flag'] == 'ambiguous_not_linked', 'entity_id'].unique())

# 2. The cohort: the quarter the business started in. 3. The 1-year check: 4 quarters later.
start_quarter = pd.to_datetime(labels['start_day'], errors='coerce').dt.to_period('Q')
labels['cohort'] = start_quarter.astype(str)
labels['one_year_quarter'] = (start_quarter + 4).astype(str)
labels['cohort_quality'] = labels['cohort'].map(COHORT_QUALITY).fillna('cannot_judge')
eligible = labels['cohort_quality'] != 'cannot_judge'

# Which businesses (entity_id) and which IDs are present in each trustworthy quarter
needed = sorted(set(labels.loc[eligible, 'one_year_quarter']))
entities_in = {q: pd.Index(use.loc[use['quarter'] == q, 'entity_id'].unique()) for q in TRUSTED}
ids_in = {q: pd.Index(use.loc[use['quarter'] == q, 'id'].unique()) for q in needed}

# 4. The label: 1 = still there after 1 year, 0 = disappeared (reason unknown)
survived = pd.Series(pd.NA, index=labels.index, dtype='Int8')
reason = pd.Series('cannot_judge', index=labels.index, dtype=object)
for q in needed:
    rows = (eligible & (labels['one_year_quarter'] == q)).to_numpy()
    businesses = labels.index[rows]
    present = businesses.isin(entities_in[q])             # under any of its IDs
    own_id_present = businesses.isin(ids_in[q])           # under its original ID (entity_id is the original ID)
    back_later = np.zeros(len(businesses), dtype=bool)    # missing at the check, but in a later quarter
    for later_q in TRUSTED[TRUSTED.index(q) + 1:]:
        back_later |= businesses.isin(entities_in[later_q])
    back_later &= ~present

    reason[rows] = np.where(present & own_id_present, 'present_after_1_year',
                   np.where(present, 'came_back_with_new_id',
                   np.where(back_later, 'missing_at_1_year_back_later', 'disappeared')))
    values = pd.array(np.where(present, 1, 0), dtype='Int8')
    values[back_later] = pd.NA                            # undecided for now: shown below, decided together
    survived[rows] = values

labels['survived_1y'] = survived
labels['label_reason'] = reason.astype('category')
del use, entities_in, ids_in, survived, reason
gc.collect()

# Check: exactly one row per business, no duplicates
print(f'Businesses in the label table: {len(labels):,} | one row each (no duplicates): {labels.index.is_unique}')
print()

# 5. Survival rate per cohort
labelled = labels[eligible]
per_cohort = labelled.groupby('cohort').agg(
    quality=('cohort_quality', 'first'),
    businesses=('survived_1y', 'size'),
    survived=('survived_1y', lambda s: int((s == 1).sum())),
    failed=('survived_1y', lambda s: int((s == 0).sum())),
    undecided=('survived_1y', lambda s: int(s.isna().sum())),
)
per_cohort['survival rate'] = (per_cohort['survived'] / (per_cohort['survived'] + per_cohort['failed'])).map('{:.1%}'.format)
print('Survival per cohort:')
print(per_cohort.to_string())
print()
print('Why each business got its label:')
print(labels['label_reason'].value_counts().to_string())
print()
print(f'Labelled businesses with an unsure ID link (ambiguous_link): {int(labelled["ambiguous_link"].sum()):,}')


## Part 5 — SSIC classification from the business name

### Cell 16 — The SSIC divisions and a 300-name test sample
Loads the official SSIC file (Gastat) and lists its divisions. Then picks 300 random businesses from the
cohorts we train on (2024 Q4, 2025 Q1, 2025 Q2), one name each, and saves them to `ssic_test_sample.csv`
(a new file on the runtime's disk). A division is suggested for each name, and a person checks every one:
the checked divisions are the answer key used to compare the classification methods.

Needs Cells 1, 8, 13 and 14 run first.

In [ ]:
SSIC_FILE = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
SSIC_PATH = DATA_DIR / 'Gastat' / SSIC_FILE
if IN_COLAB and not SSIC_PATH.exists():
    import gdown
    SSIC_PATH.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id='1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ', output=str(SSIC_PATH), quiet=True)

# The SSIC sheet starts with a title; the real header is the row that starts with "رمز الباب"
raw = pd.read_excel(SSIC_PATH, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
ssic = raw.iloc[header_row + 1:].copy()
ssic.columns = ['section_code', 'section_name', 'division_code', 'division_name', 'group_code', 'group_name',
                'class_code', 'class_name', 'activity_code', 'activity_name', 'activity_description',
                'supervising_authority', 'licensing_authority']
ssic = ssic.dropna(subset=['activity_code']).apply(lambda column: column.str.strip())

divisions = (ssic.groupby(['section_code', 'section_name', 'division_code', 'division_name'], sort=False)
             .size().rename('activities').reset_index())
print(f'SSIC: {len(ssic):,} activities | {ssic["section_code"].nunique()} sections | {len(divisions)} divisions')
print()

# 300 random businesses from the cohorts we will train on (one name per business: its latest name)
pool = pd.Series(labels.index[labels['cohort_quality'].isin(['clean', 'start_incomplete'])])
sample_ids = pool.sample(300, random_state=42)
rows = mc.loc[mc['entity_id'].isin(sample_ids) & (mc['record_type'] == 'رئيسي'), ['entity_id', 'quarter', 'name', 'city']]
rows = rows.assign(order=rows['quarter'].map({q: i for i, q in enumerate(QUARTERS)}).astype(int))
sample = (rows.sort_values('order').drop_duplicates('entity_id', keep='last')
          [['entity_id', 'name', 'city']].sort_values('entity_id').reset_index(drop=True))
sample.index = sample.index + 1
sample['suggested_division'] = ''     # filled in next, then checked by a person
sample['correct_division'] = ''       # the answer key: the division a person confirms

sample.to_csv(DATA_DIR / 'ssic_test_sample.csv', index_label='no', encoding='utf-8-sig')
print(f'Saved {len(sample)} names to {DATA_DIR / "ssic_test_sample.csv"}')
print()
for no, row in sample.iterrows():
    print(f'{no}|{row["entity_id"]}|{row["name"]}|{row["city"]}')


### Cell 17 — Suggest a division for the 300 test names
The MC files have **no activity or SSIC column**, so the SSIC code is found from the business name with the
rules in `ml/src/isic_classifier.py`. The classifier first checks itself on a few known names, then fills
`ssic_section`, `suggested_division`, `ssic_flag` and `ssic_candidates` for the 300 names and saves
`ssic_test_sample_suggested.csv`.

**Then a person checks every name** (open the file in Excel or Google Sheets):
- Write the right 2-digit SSIC division in `correct_division` (e.g. `56` for restaurants, `47` for retail).
  Look the code up in the SSIC file, not in the suggestion.
- If the name alone does not tell the activity (e.g. "مؤسسة سالم محمد العتيبي"), write `?`.
- Do not change the other columns. Save as `ssic_test_sample_checked.csv` in the data folder
  (on Colab: upload it to `/content/darb-data`).

Needs Cell 16 run first.

In [ ]:
import sys
import importlib

# Where isic_classifier.py is: next to the notebook on Colab (upload it), ml/src on a laptop
SRC_DIR = Path('/content') if IN_COLAB else Path.cwd().parent / 'src'
if not (SRC_DIR / 'isic_classifier.py').exists():
    raise FileNotFoundError(f'Put isic_classifier.py in {SRC_DIR} first (on Colab: Files panel -> Upload).')
sys.path.insert(0, str(SRC_DIR))
import isic_classifier
importlib.reload(isic_classifier)                 # picks up rule changes without a restart
from isic_classifier import classify_tuple, RESULT, self_test

# The classifier must pass its own examples before it is used
assert self_test(), 'The classifier failed its self-test: fix isic_classifier.py first.'

# The SSIC file writes sections with Arabic letters (أ، ب، ج …), the classifier with ISIC letters (A, B, C …).
# Both follow the same order, so the n-th section of the file is the n-th letter.
arabic_sections = list(dict.fromkeys(divisions['section_code']))
SECTION_LETTER = dict(zip(arabic_sections, 'ABCDEFGHIJKLMNOPQRSTU'))
divisions['section_letter'] = divisions['section_code'].map(SECTION_LETTER)
letter_of_division = dict(zip(divisions['division_code'].str.zfill(2), divisions['section_letter']))
assert [letter_of_division[d] for d in ['01', '10', '41', '47', '56', '86']] == ['A', 'C', 'F', 'G', 'I', 'Q'], \
    'The Arabic section letters do not line up with A, B, C …: check the SSIC file.'
print('SSIC sections:', ', '.join(f'{a} = {l}' for a, l in SECTION_LETTER.items()))

# Suggest a division for every name of the test sample
suggested = pd.DataFrame([classify_tuple(n) for n in sample['name']], index=sample.index, columns=RESULT)
sample['ssic_section'] = suggested['ssic_section']
sample['suggested_division'] = suggested['ssic_division']
sample['ssic_flag'] = suggested['ssic_flag']
sample['ssic_candidates'] = suggested['ssic_candidates']
sample = sample[['entity_id', 'name', 'city', 'ssic_section', 'suggested_division', 'ssic_flag',
                 'ssic_candidates', 'correct_division']]

sample.to_csv(DATA_DIR / 'ssic_test_sample_suggested.csv', index_label='no', encoding='utf-8-sig')
print(f'Saved {len(sample)} names to {DATA_DIR / "ssic_test_sample_suggested.csv"}')
print()
print('Flags in the sample:')
print(sample['ssic_flag'].value_counts().to_string())


### Cell 18 — How accurate is the classifier? (only counts)
Reads the checked sample `ssic_test_sample_checked.csv` and compares the classifier with the answer key:
- **Coverage:** share of names that got a section (not flagged `ambiguous` / `unclassified`).
- **Section accuracy:** of those, how many have the right section.
- **Division accuracy:** of the names flagged `ok`, how many have the right division.
- **Missed:** names the classifier flagged, but a person could classify.

The wrong ones are listed, to improve the rules in `isic_classifier.py`. After changing a rule, run Cells 17 and 18
again (the answer key stays the same). Write the final numbers in the notes at the end of Part 5.

In [ ]:
CHECKED = DATA_DIR / 'ssic_test_sample_checked.csv'
# sep=None: works for a comma file (saved from Excel) and a tab file (pasted with %%writefile)
checked = pd.read_csv(CHECKED, dtype=str, encoding='utf-8-sig', sep=None, engine='python').fillna('')
for column in ['ssic_section', 'suggested_division', 'correct_division']:
    checked[column] = checked[column].str.strip()

# The answer key: a 2-digit SSIC division, or "?" when a person cannot tell from the name either
is_code = checked['correct_division'].str.fullmatch(r'\d{1,2}')
checked.loc[is_code, 'correct_division'] = checked.loc[is_code, 'correct_division'].str.zfill(2)
section_of = letter_of_division                            # division -> section letter (Cell 17)
empty = checked['correct_division'] == ''
cannot_tell = checked['correct_division'] == '?'
not_a_code = ~empty & ~cannot_tell & ~checked['correct_division'].isin(section_of)
print(f'Checked file: {len(checked)} names | not filled in: {empty.sum()} | "?": {cannot_tell.sum()} | '
      f'not an SSIC division: {not_a_code.sum()}')
if empty.any() or not_a_code.any():
    print('Fix these rows first:')
    print(checked.loc[empty | not_a_code, ['no', 'name', 'correct_division']].to_string(index=False))

checked['correct_section'] = checked['correct_division'].map(section_of).fillna('')
judged = checked[checked['correct_section'] != '']          # names a person could classify
given = judged['ssic_section'] != ''                        # names the classifier gave a section

section_right = judged['ssic_section'] == judged['correct_section']
division_rows = judged['ssic_flag'] == 'ok'
division_right = judged['suggested_division'] == judged['correct_division']

print()
print(f'Names a person could classify: {len(judged)} of {len(checked)}')
print(f'Coverage   (classifier gave a section):          {given.mean():.1%}')
print(f'Section accuracy  (where a section was given):   {section_right[given].mean():.1%}')
print(f'Division accuracy (where the flag is "ok"):      {division_right[division_rows].mean():.1%}')
print(f'Missed (flagged, but a person could tell):       {(~given).sum()}')

print()
print('By flag:')
by_flag = judged.assign(section_right=section_right, division_right=division_right).groupby('ssic_flag').agg(
    names=('name', 'size'), section_right=('section_right', 'mean'), division_right=('division_right', 'mean'))
print(by_flag.to_string(float_format='{:.1%}'.format))

print()
print('Wrong sections (to fix the rules):')
wrong = judged[given & ~section_right]
print(wrong[['no', 'name', 'ssic_section', 'suggested_division', 'ssic_candidates', 'correct_division']].to_string(index=False))


### Cell 19 — Classify every business (adds columns, deletes nothing)
1. **One name per business:** the latest name of its main record (the same choice as the test sample).
2. Each name gets a section, a division and an `ssic_flag` from the classifier.
3. **Consistency:** a business gets one code for all quarters. If its names in different quarters point to
   different sections, the latest name's code is kept but the flag becomes `names_disagree`.
4. **Check:** every code given must exist in the official SSIC file (Cell 16).
5. The columns are added to `labels` (one row per business) and saved to `ssic_by_business.csv`.

`ssic_flag`: `ok`, `division_unclear`, `low_confidence`, `ambiguous`, `unclassified`, `names_disagree`.
Only `ok` and `division_unclear` have a sure section. The other flags are kept as they are, never filled in by guessing.

Needs Cells 1, 8, 13, 14, 16 and 17 run first.

In [ ]:
from isic_classifier import classify_tuple, RESULT

# 1. One name per business: its latest name (main records only, like the label table and the test sample)
QUARTER_ORDER = {q: i for i, q in enumerate(QUARTERS)}
use = mc.loc[(mc['record_type'] == 'رئيسي')
             & mc['link_flag'].isin(['same_id', 'linked_new_id', 'ambiguous_not_linked'])
             & mc['name'].notna(), ['entity_id', 'quarter', 'name']]
use = use.assign(order=use['quarter'].map(QUARTER_ORDER).astype(int))
names = use.drop_duplicates(['entity_id', 'name'])             # every different name a business had
del use
gc.collect()

# 2. Classify each different name once (many businesses share a name pattern; this saves time)
result_of = {name: classify_tuple(name) for name in names['name'].unique()}
print(f'Different names classified: {len(result_of):,}')

latest = names.sort_values('order').drop_duplicates('entity_id', keep='last').set_index('entity_id')['name']
ssic_by_business = pd.DataFrame([result_of[n] for n in latest], index=latest.index, columns=RESULT)

# 3. Consistency: one code per business in every quarter. If its names point to different sections,
#    the latest name's code is kept but the business is flagged "names_disagree".
section_per_name = names.assign(section=names['name'].map(lambda n: result_of[n][0]))
section_per_name = section_per_name[section_per_name['section'] != '']
sections_per_business = section_per_name.groupby('entity_id')['section'].agg(lambda s: '|'.join(sorted(set(s))))
disagree = sections_per_business[sections_per_business.str.contains('|', regex=False)]
ssic_by_business.loc[disagree.index, 'ssic_flag'] = 'names_disagree'
ssic_by_business.loc[disagree.index, 'ssic_candidates'] = disagree
del names, section_per_name, sections_per_business, result_of
gc.collect()

# 4. Every code given must exist in the official SSIC file (section letters from Cell 17)
official = set(zip(divisions['section_letter'], divisions['division_code'].str.zfill(2)))
given = ssic_by_business[ssic_by_business['ssic_division'] != '']
not_official = given[[pair not in official for pair in zip(given['ssic_section'], given['ssic_division'])]]
print(f'Codes not found in the SSIC file: {len(not_official):,}', '(fix the rule)' if len(not_official) else '')
if len(not_official):
    print(not_official[['ssic_section', 'ssic_division', 'ssic_label']].value_counts().to_string())

# The official Arabic names of the section and division
section_names = dict(zip(divisions['section_letter'], divisions['section_name']))
division_names = dict(zip(divisions['division_code'].str.zfill(2), divisions['division_name']))
ssic_by_business['ssic_section_name'] = ssic_by_business['ssic_section'].map(section_names).fillna('')
ssic_by_business['ssic_division_name'] = ssic_by_business['ssic_division'].map(division_names).fillna('')

# 5. Add to the label table (one row per business). A business with no name at all stays unclassified.
SSIC_COLUMNS = RESULT + ['ssic_section_name', 'ssic_division_name']
labels = labels.drop(columns=SSIC_COLUMNS, errors='ignore').join(ssic_by_business[SSIC_COLUMNS])
labels['ssic_flag'] = labels['ssic_flag'].fillna('unclassified')
labels[SSIC_COLUMNS] = labels[SSIC_COLUMNS].fillna('')
labels['ssic_flag'] = labels['ssic_flag'].astype('category')
del ssic_by_business, latest, given, not_official
gc.collect()

# 6. Counts
labelled = labels[labels['cohort_quality'] != 'cannot_judge']
flags = pd.DataFrame({'all businesses': labels['ssic_flag'].value_counts(),
                      'labelled cohorts': labelled['ssic_flag'].value_counts()})
flags['% of labelled'] = (flags['labelled cohorts'] / len(labelled)).map('{:.1%}'.format)
print()
print('ssic_flag (nothing is guessed: every business without a sure code is flagged):')
print(flags.to_string())
print()
print('Labelled businesses per section (flag ok or division_unclear):')
sure = labelled[labelled['ssic_flag'].isin(['ok', 'division_unclear'])]
print(sure.groupby(['ssic_section', 'ssic_section_name']).size().sort_values(ascending=False).to_string())

# 7. Save (a new file on the runtime's disk, never in Git)
labels[['cohort', 'cohort_quality', 'survived_1y'] + SSIC_COLUMNS].to_csv(
    DATA_DIR / 'ssic_by_business.csv', index_label='entity_id', encoding='utf-8-sig')
print()
print(f'Saved {len(labels):,} businesses to {DATA_DIR / "ssic_by_business.csv"}')


### Notes — SSIC mapping (fill in after running Cells 17–19)
- **Method:** the SSIC code is found from the business name (keyword rules, `ml/src/isic_classifier.py`),
  because the MC files have no activity column.
- **Test sample (300 names):** coverage __%, section accuracy __%, division accuracy __%. Checked by: __.
- **All labelled businesses:** `ok` __%, `division_unclear` __%, `low_confidence` __%, `ambiguous` __%,
  `unclassified` __%, `names_disagree` __%.
- **Known limits:** names that do not say the activity (personal names) cannot be classified; names with
  no Arabic letters are `unclassified`; a business is classified by its latest name.